In [1]:
# This cell is removed with the tag: "remove-input"
# As such, it will not be shown in documentation

import warnings
warnings.filterwarnings('ignore')


(Tutorial_Potential_energy_minimization)=
# Potential energy minimization

*Relaxing molecular structures to local potential energy minima using molecular mechanics optimizers.*

The function {func}`molsysmt.molecular_mechanics.potential_energy_minimization` performs local L-BFGS optimization of a prepared force-field model. It can reduce geometric strain; completion does not guarantee the nearest/global minimum, equilibrium or physical stability.

:::{versionadded} 1.0.0
:::

:::{admonition} API documentation
:class: dropdown

Follow this link for a detailed description of the input arguments, raised errors, and returned objects of this function: {func}`molsysmt.molecular_mechanics.potential_energy_minimization`.
:::


:::{note}
:class: dropdown

**Molecular Mechanics Backend Engine**

Molecular mechanics and force evaluations in MolSysMT are executed using backend simulation engines (currently **OpenMM** by default). MolSysMT intentionally unifies the Pythonic interface and data structures while delegating energy evaluations, parameterizations, and gradient calculations to high-performance, battle-tested molecular simulation engines.
:::

Execution platform names belong to the selected engine. OpenMM accepts `Reference`, `CPU`, `CUDA`, `OpenCL` and installed plugin names, subject to registration and device readiness. The `platform` argument controls newly constructed contexts; an existing Context/Simulation retains its platform. A requested platform is never replaced after failure. See {ref}`OpenMM platform discovery <Tutorial_OpenMM_Available_platforms>` for registration and distinct error outcomes.


## Basic usage

Let's load the Trp-Cage miniprotein and minimize its potential energy:


In [2]:
import molsysmt as msm
import pyunitwizard as puw


In [3]:
molsys = msm.convert(msm.systems['Trp-Cage']['1l2y.h5msm'], structure_indices=0)


:::{note}
:class: dropdown

**Demo Systems Catalog**

This tutorial uses demonstration datasets provided by MolSysMT. To explore the full catalog of bundled systems, forms, and file paths, visit the {ref}`Demo Systems <user-foundations-entrance-demo-systems>` guide.
:::


For a native `MolSys`, the default call returns an independent system. Retain that result (`molsys_min` below) and use it for the after-energy calculation. `molsys` retains the original coordinates. Both energies must use the same chemical/force-field model and explicit units when exporting numbers.

The public method is L-BFGS, using the selected engine's default stopping settings. There is no `tolerance` or iteration-count argument in this API. Under constraints, a successful search does not imply that every atom force vanishes. Prepare chemistry and parameters before optimization; energy reduction alone is not evidence of binding or thermodynamic stability.


In [4]:
u_pre = msm.molecular_mechanics.get_potential_energy(molsys)
molsys_min = msm.molecular_mechanics.potential_energy_minimization(molsys)
u_post = msm.molecular_mechanics.get_potential_energy(molsys_min)

print('Energy before minimization:', u_pre)
print('Energy after minimization:', u_post)


Energy before minimization: -1233.257063435885 kilojoule / mole
Energy after minimization: -2132.360876052932 kilojoule / mole


## In-place coordinate minimization

Setting `in_place=True` deliberately updates the native input coordinates and returns `None`. Keep the existing `molsys` variable; do not assign that `None` back over it. The following call starts from the original system retained above:


In [5]:
result = msm.molecular_mechanics.potential_energy_minimization(molsys, in_place=True)
assert result is None
print('Minimized in-place energy:', msm.molecular_mechanics.get_potential_energy(molsys))


Minimized in-place energy: -2132.360876052932 kilojoule / mole


## Existing engine inputs

An independent request first uses the source form's public copy contract. Copies of existing OpenMM Context/Simulation objects are currently unsupported, so the operation raises **before optimization** and preserves the input. An explicit `to_form` or a composite containing an engine object does not bypass this check. No partial State snapshot is used to reconstruct a different model/platform.

The following Reference context illustrates an unchanged source after rejection. The existing native `MolSys` examples above remain the supported independent workflow.


In [6]:
context = msm.convert(molsys_min, to_form='openmm.Context', platform='Reference')
source_coordinates = puw.get_value(msm.get(context, coordinates=True), to_unit='nm').copy()
try:
    msm.molecular_mechanics.potential_energy_minimization(context)
except msm.NotImplementedMethodError as error:
    print(type(error).__name__)
else:
    raise AssertionError('Existing engine copying is not implemented')
assert (puw.get_value(msm.get(context, coordinates=True), to_unit='nm') == source_coordinates).all()


ERROR: This method is not implemented yet: independent copy of openmm.Context. Check alternative methods or wait for an update. Docs: https://www.uibcdf.org/MolSysMT (Hint: Check alternative methods or wait for an update. Docs: https://www.uibcdf.org/MolSysMT)


NotImplementedMethodError


For deliberate editing of that existing engine object, use `in_place=True`. It keeps the actual model/platform and returns `None`. This mode does not promise rollback if the engine itself fails during an explicitly requested edit.


In [7]:
result = msm.molecular_mechanics.potential_energy_minimization(context, in_place=True)
print('Returns None:', result is None)
print('Actual platform:', context.getPlatform().getName())


Returns None: True
Actual platform: Reference


:::{seealso}
:class: dropdown

**Related Tools & References**

- {ref}`Convert <Tutorial_Convert>`: Convert molecular systems with {func}`molsysmt.basic.convert`.
- {ref}`Get potential energy <Tutorial_Get_potential_energy>`: Calculate potential energy with {func}`molsysmt.molecular_mechanics.get_potential_energy`.
- {ref}`Get forces <Tutorial_Get_forces>`: Extract atomic forces with {func}`molsysmt.molecular_mechanics.get_forces`.
:::
